<a href="https://colab.research.google.com/github/Innovatewithapple/NeuralNetwork/blob/main/NLPTransformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Text → Tokens → Embedding → Attention → Output words**

In [38]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input,Dense,Embedding,Layer
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [39]:
decoder_output

array([[[3],
        [2],
        [0],
        [0]],

       [[4],
        [5],
        [2],
        [0]],

       [[6],
        [7],
        [8],
        [2]]], dtype=int32)

In [23]:
input_texts = ["Hello", "How are you", "I love programming"]
target_texts = ["Hola", "Cómo estás", "Me encanta programar"]

target_texts = ["startseq "+ t +" endseq" for t in target_texts]

In [24]:
target_texts

['startseq Hola endseq',
 'startseq Cómo estás endseq',
 'startseq Me encanta programar endseq']

in below code:

fit_on_text just create numbers of each word from the sentence like:

hello -> 1

world -> 2

mihir -> 3


texts_to_sequence just create a dicitonay sequence like:

hello world -> [1,2]

hello mihir -> [1,3]


input_vocab_size means we rearrange the dictonary and make it for all equal:

first dictionary is [1,2,3]

second dictionary is [4]

so vocab just do: [4,0,0]

In [25]:
#tokansization
input_tokenizer = Tokenizer()
input_tokenizer.fit_on_texts(input_texts)
input_sequences = input_tokenizer.texts_to_sequences(input_texts)

output_tokenizer = Tokenizer()
output_tokenizer.fit_on_texts(target_texts)
output_sequences = output_tokenizer.texts_to_sequences(target_texts)

input_vocab_size = len(input_tokenizer.word_index) + 1
output_vocab_size = len(output_tokenizer.word_index) + 1

In [26]:
input_tokenizer.word_index

{'hello': 1, 'how': 2, 'are': 3, 'you': 4, 'i': 5, 'love': 6, 'programming': 7}

In [27]:
output_tokenizer.word_index

{'startseq': 1,
 'endseq': 2,
 'hola': 3,
 'cómo': 4,
 'estás': 5,
 'me': 6,
 'encanta': 7,
 'programar': 8}

this below code basically set equal lenths of dictonairy, and the post means added those paddings of 0 at the last. because in transformer thats what we do

In [28]:
max_input_len = max(len(seq) for seq in input_sequences)
max_output_len = max(len(seq) for seq in output_sequences)

input_sequences = pad_sequences(input_sequences,maxlen=max_input_len,padding='post')
output_sequences = pad_sequences(output_sequences,maxlen=max_output_len,padding='post')

Basically we predict next by previous word so here the previous word become input and output will be the next word. and that output will be input for next.

In [29]:
output_sequences

array([[1, 3, 2, 0, 0],
       [1, 4, 5, 2, 0],
       [1, 6, 7, 8, 2]], dtype=int32)

In [30]:
output_sequences[:,:-1]

array([[1, 3, 2, 0],
       [1, 4, 5, 2],
       [1, 6, 7, 8]], dtype=int32)

In [31]:
output_sequences[:,1:]

array([[3, 2, 0, 0],
       [4, 5, 2, 0],
       [6, 7, 8, 2]], dtype=int32)

In [32]:
#decoder input/output.
# decoder_input = output_sequences[:,:-1]
# decoder_output = output_sequences[:,1:]
# decoder_output = np.expand_dims(decoder_output, -1)

In [33]:
# The input is the whole sentence at once, and every single word performs this "matching" process against every other word (including itself).
# Here is how that works for "Apple" in the sentence: "I ate a red Apple."
# 1. The Input
# The model receives the whole sentence. It creates a Query, Key, and Value for every single word simultaneously.
# 2. The Matching (Key Part)
# You are right—the Query of "Apple" is compared against the Key of every word in the sentence. It looks like a round-robin tournament:
# Apple (Q) vs. I (K) → Low match
# Apple (Q) vs. ate (K) → Medium match
# Apple (Q) vs. a (K) → Low match
# Apple (Q) vs. red (K) → High match!
# Apple (Q) vs. Apple (K) → High match! (A word always pays attention to itself).

In [34]:
#attention layer
#initialization
class Attention(Layer):
  def __init__(self,d_model):
    super().__init__()
    self.Wq = Dense(d_model)
    self.Wk = Dense(d_model)
    self.Wv = Dense(d_model)

  def call(self,q,k,v):
    Q = self.Wq(q)
    K = self.Wk(k)
    V = self.Wv(v)

    scores = tf.matmul(Q,K,transpose_b=True)
    dk = tf.cast(tf.shape(K)[-1], tf.float32) # it will take the d_model thats all
    scores = scores / tf.sqrt(dk)

    weights = tf.nn.softmax(scores) # convert numbers into probablities
    output = tf.matmul(weights, V)

    return output

In [35]:
encoder_inputs = Input(shape=(max_input_len,))
encoder_inputs

<KerasTensor shape=(None, 3), dtype=float32, sparse=False, ragged=False, name=keras_tensor_24>

In [36]:
decoder_output

array([[[3],
        [2],
        [0],
        [0]],

       [[4],
        [5],
        [2],
        [0]],

       [[6],
        [7],
        [8],
        [2]]], dtype=int32)

In [20]:
encoder_inputs = Input(shape=(max_input_len,))
x = Embedding(input_vocab_size, 64)(encoder_inputs)

encoder_output = Attention(64)(x, x, x)


decoder_inputs = Input(shape=(max_output_len - 1,))
y = Embedding(output_vocab_size, 64)(decoder_inputs)

y = Attention(64)(y, y, y)
y = Attention(64)(y, encoder_output, encoder_output)

outputs = Dense(output_vocab_size, activation='softmax')(y)

model = Model([encoder_inputs, decoder_inputs], outputs)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    [input_sequences, decoder_input],
    decoder_output,
    epochs=200
)

Epoch 1/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - accuracy: 0.1667 - loss: 2.1931
Epoch 2/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.2500 - loss: 2.1794
Epoch 3/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.2500 - loss: 2.1657
Epoch 4/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.2500 - loss: 2.1518
Epoch 5/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.3333 - loss: 2.1377
Epoch 6/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.3333 - loss: 2.1233
Epoch 7/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.3333 - loss: 2.1085
Epoch 8/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.3333 - loss: 2.0931
Epoch 9/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step - accuracy: 0.3333 - loss: 2.0772
Epoch 10/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.3333 - loss: 2.0606
Epoch 11/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.3333 - loss: 2.0432
Epoch 12/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.3333 - los